# Priority 2 — IEEE-CIS Dataset: EDA and Pipeline Validation

**Project:** Fraud AI Investigator — MENA Fintech Portfolio  
**Notebook:** `notebooks/priority2_ieee_cis.ipynb`  
**Last updated:** 2026-05

---

## Objective

Validate and analyse the IEEE-CIS dataset integration. This notebook answers:

1. **Is the IEEE-CIS data loading correctly?** — schema validation, record counts
2. **How does the real fraud distribution compare to synthetic?** — amount, time, device
3. **Does rule engine performance change on real data?** — precision/recall vs synthetic baseline
4. **What does the data source status look like?** — live API check

## Why IEEE-CIS matters

The synthetic Faker dataset has artificially clean distributions — fraud and non-fraud
are perfectly separated by amount and country. IEEE-CIS has real-world characteristics:
- 3.5% fraud rate (vs 20% synthetic) — much harder classification problem
- Skewed transaction amounts — most transactions are small, fraud can be any size
- Missing values (0.1–99% per feature) — real systems must handle incomplete data
- 590k transactions across a full year — temporal patterns matter

Precision/recall numbers against IEEE-CIS are credible in interviews.
Against synthetic data they are illustrative only.

## Prerequisites

```bash
# Option A — with real data (recommended)
# 1. Download from kaggle.com/competitions/ieee-fraud-detection/data
# 2. Place train_transaction.csv and train_identity.csv in app/data/ieee_cis/
# 3. Run processor:
uv run python scripts/load_ieee_data.py

# Option B — API works with synthetic data fallback (no download needed)
# The system automatically falls back to synthetic data

# Start API:
uv run uvicorn app.main:app --reload

# Launch notebook:
uv run jupyter notebook notebooks/priority2_ieee_cis.ipynb
```

In [ ]:
import json
import sys
import warnings
from datetime import datetime
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import requests
from requests.exceptions import ConnectionError as ReqConnErr

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))
warnings.filterwarnings('ignore', category=DeprecationWarning)

plt.style.use('seaborn-v0_8-whitegrid')
pd.set_option('display.max_columns', 20)

BASE_URL        = 'http://localhost:8000'
SCREENSHOTS_DIR = PROJECT_ROOT / 'doc' / 'Screenshots'
SCREENSHOTS_DIR.mkdir(parents=True, exist_ok=True)
DATA_DIR        = PROJECT_ROOT / 'app' / 'data'

print(f'Project root : {PROJECT_ROOT}')
print(f'Run timestamp: {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}')

---
## Section 1 — API health and data source check

In [ ]:
try:
    health = requests.get(f'{BASE_URL}/health', timeout=5).json()
except ReqConnErr:
    raise RuntimeError('API not running — start with: uv run uvicorn app.main:app --reload')

assert health['status'] == 'ok'
print('API HEALTH')
print(f"  Status  : {health['status']}")
print(f"  Version : {health['version']}")

# Check data source status
ds = requests.get(f'{BASE_URL}/v1/alerts/datasource').json()
print()
print('DATA SOURCE STATUS')
print(f"  Active source    : {ds['active_source']}")
print(f"  IEEE-CIS available: {ds['ieee_cis_available']}")
print(f"  IEEE-CIS count   : {ds['ieee_cis_count']:,}")
print(f"  Synthetic count  : {ds['synthetic_count']:,}")
if ds.get('download_instructions'):
    print(f"\n  To add real data: {ds['download_instructions']}")

---
## Section 2 — Load and inspect the processed IEEE-CIS data

If IEEE-CIS is not downloaded yet, this section uses synthetic data and notes the difference.

In [ ]:
ieee_path  = DATA_DIR / 'ieee_cis' / 'ieee_transactions.json'
synth_path = DATA_DIR / 'transactions.json'

if ieee_path.exists():
    with open(ieee_path) as f:
        raw = json.load(f)
    source_label = 'IEEE-CIS (real data)'
    print('Loading IEEE-CIS processed data...')
else:
    with open(synth_path) as f:
        raw = json.load(f)
    source_label = 'Synthetic Faker data (fallback)'
    print('IEEE-CIS not downloaded — using synthetic fallback')
    print('Download: https://www.kaggle.com/competitions/ieee-fraud-detection/data')

df = pd.DataFrame(raw)
df['amount_aed'] = df['amount_aed'].astype(float)

fraud     = df[df['is_flagged'] == True]
non_fraud = df[df['is_flagged'] == False]

print(f'\nDATA SUMMARY ({source_label})')
print('=' * 55)
print(f'Total transactions : {len(df):,}')
print(f'Fraud              : {len(fraud):,} ({len(fraud)/len(df):.1%})')
print(f'Non-fraud          : {len(non_fraud):,} ({len(non_fraud)/len(df):.1%})')
print(f'Amount range (AED) : {df["amount_aed"].min():,.0f} – {df["amount_aed"].max():,.0f}')
print(f'Fraud mean (AED)   : {fraud["amount_aed"].mean():,.0f}')
print(f'Non-fraud mean (AED): {non_fraud["amount_aed"].mean():,.0f}')

---
## Section 3 — Amount distribution: fraud vs non-fraud

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, label, data, color in [
    (axes[0], 'Non-fraud', non_fraud, '#4CAF50'),
    (axes[1], 'Fraud',     fraud,     '#F44336'),
]:
    ax.hist(data['amount_aed'], bins=30, color=color, alpha=0.8, edgecolor='white')
    ax.axvline(x=40000, color='crimson', linestyle='--', linewidth=1.5,
               label='AED 40k CBUAE threshold')
    ax.set_title(f'{label} transactions — AED amount distribution', fontsize=12)
    ax.set_xlabel('Transaction amount (AED)')
    ax.set_ylabel('Count')
    ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1000:.0f}k'))
    ax.legend(fontsize=9)

plt.suptitle(f'Amount Distribution — {source_label}', y=1.02, fontsize=13, fontweight='bold')
plt.tight_layout()
save_path = SCREENSHOTS_DIR / '07_ieee_amount_distribution.png'
plt.savefig(save_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Chart saved: {save_path}')

---
## Section 4 — Rule engine performance on real data

In [ ]:
HIGH_VALUE_THRESHOLD = 40_000
HIGH_RISK_COUNTRIES  = {'IR','KP','SY','MM','YE','SD','PK','NG','HT','LA','TZ','CM','CD','AO','CU','VE','BY','LY','RU'}

rule_hv   = df['amount_aed'] > HIGH_VALUE_THRESHOLD
rule_sc   = df['country'].isin(HIGH_RISK_COUNTRIES) if 'country' in df.columns else pd.Series([False]*len(df))
any_fired = rule_hv | rule_sc
truth     = df['is_flagged']

tp = int((any_fired & truth).sum())
fp = int((any_fired & ~truth).sum())
fn = int((~any_fired & truth).sum())
tn = int((~any_fired & ~truth).sum())

precision = tp / (tp + fp) if (tp + fp) > 0 else 0
recall    = tp / (tp + fn) if (tp + fn) > 0 else 0
f1        = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0

print(f'RULE ENGINE PERFORMANCE ON {source_label.upper()}')
print('=' * 50)
print(f'Rules: amount > AED 40k | country in FATF list')
print()
print(f'True Positives  (caught fraud)  : {tp}')
print(f'False Positives (false alarms)  : {fp}')
print(f'False Negatives (missed fraud)  : {fn}')
print(f'True Negatives  (correctly clear): {tn}')
print()
print(f'Precision : {precision:.1%}')
print(f'Recall    : {recall:.1%}')
print(f'F1 Score  : {f1:.1%}')
print()
print('Note: LLM triage (Phase 3) filters false positives before analyst review.')

---
## Section 5 — Generate alerts via API using real data

In [ ]:
gen = requests.post(
    f'{BASE_URL}/v1/alerts/generate',
    json={'limit': 20, 'source': 'auto'},
    timeout=30,
).json()

print('ALERT GENERATION')
print(f"  Alerts created : {gen['alerts_created']}")
print(f"  Data source    : {gen.get('data_source', 'unknown')}")
print(f"  Message        : {gen['message']}")

---
## Section 6 — Completion checklist

In [ ]:
print('PRIORITY 2 NOTEBOOK — COMPLETION CHECKLIST')
print('=' * 55)

checks = {
    'API health check passed'         : health['status'] == 'ok',
    'Data source status endpoint works': 'active_source' in ds,
    'Transactions loaded'             : len(df) > 0,
    'Amount distribution chart saved' : (SCREENSHOTS_DIR / '07_ieee_amount_distribution.png').exists(),
    'Rule engine metrics computed'    : precision > 0 or recall > 0,
    'Alerts generated via API'        : gen.get('alerts_created', 0) >= 0,
    'IEEE-CIS data available'         : ds['ieee_cis_available'],
}

all_passed = True
for label, passed in checks.items():
    print(f"  {'✓' if passed else '✗'}  {label}")
    if not passed:
        all_passed = False

print()
if all_passed:
    print('All checks passed — Priority 2 notebook complete ✓')
else:
    print('Some checks failed — see sections above.')
    if not ds['ieee_cis_available']:
        print('\n  To add real IEEE-CIS data:')
        print('  1. https://www.kaggle.com/competitions/ieee-fraud-detection/data')
        print('  2. Place CSVs in app/data/ieee_cis/')
        print('  3. uv run python scripts/load_ieee_data.py')

print(f'\nCompleted: {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}')